# Preparação dos Dados (Entrega 2)

Este notebook **dá continuidade ao tratamento da Entrega 1** (`src/Entrega 1/Projeto_Interdisciplinar_Ciência_de_Dados.ipynb`)

- **Entrada:** `Demonstrativo Fecap v3.csv` (CTI Global)
- **Saída:** `src/Staging/base_larga.parquet`

**Convenções da base (informadas pela CTI):** Ano 1 = 2027 ... Ano 12 = 2038; os 1.200 cenários são simulações de Monte Carlo da mesma concessão; o Ano 12 é o encerramento da concessão, por isso várias contas não existem nele. Os sinais da base são mantidos (passivo e patrimônio líquido vêm negativos).

## Bibliotecas e configuração

Variáveis de configuração:

- `CAMINHO_CSV`: arquivo da CTI, lido direto da pasta `Processing`.
- `PASTA_SAIDA`: `src/Staging`, quando o notebook roda a partir de `src/Notebooks`.
- `ANO_BASE`: convenção da CTI, Ano 1 = 2027, então o ano calendário é `ANO_BASE + ano_num`.

In [13]:
import os
import re
import unicodedata

import pandas as pd

PASTA_ENTRADA = "../Processing"
CAMINHO_CSV = os.path.join(PASTA_ENTRADA, "Demonstrativo Fecap v3.csv")

PASTA_SAIDA = "../Staging"
os.makedirs(PASTA_SAIDA, exist_ok=True)

ANO_BASE = 2026

## Etapa 2 - Coletar dados iniciais

In [14]:
colunas = ["ano", "cenario", "conta", "valor"]

df = pd.read_csv(
    CAMINHO_CSV,
    sep=";",
    encoding="latin-1",
    header=None,
    names=colunas,
    dtype=str,
)

print("Formato bruto:", df.shape)

Formato bruto: (1002000, 4)


## Etapa 5 - Selecionando os dados

Linhas sem `conta` são separadores/controles do arquivo (3 por cenário-ano, 43.200 no total) e não pertencem a nenhuma conta, então são removidas, como na Entrega 1.

In [15]:
registros_antes = len(df)

df = df[df["conta"].notna()].copy()

print("Registros antes da seleção:", registros_antes)
print("Registros depois da seleção:", len(df))
print("Registros removidos no processo:", registros_antes - len(df))

Registros antes da seleção: 1002000
Registros depois da seleção: 958800
Registros removidos no processo: 43200


## Etapa 6 - Limpando e uniformizando os dados

- Espaços duplicados nos nomes das contas são uniformizados (ex.: `BAL  - Capital Social`).
- Espaços nas pontas de cenário e ano são removidos.
- `valor` está em texto no formato brasileiro (`1.234,56`) e é convertido para número.

In [16]:
df["conta"] = df["conta"].str.replace(r"\s+", " ", regex=True).str.strip()
df["cenario"] = df["cenario"].str.strip()
df["ano"] = df["ano"].str.strip()

df["valor"] = (
    df["valor"]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

print("Formato após a limpeza:", df.shape)

Formato após a limpeza: (958800, 4)


## Etapa 7 - Derivando e integrando os dados

`ano_num` e `cenario_num` são extraídos do texto: `Ano 3` vira 3 e `Total Cen_00012` vira 12.

In [19]:
df["ano_num"] = df["ano"].str.extract(r"(\d+)", expand=False).astype(int)
df["cenario_num"] = df["cenario"].str.extract(r"(\d+)", expand=False).astype(int)

### Nome da coluna de cada conta

As colunas seguem a ordem que aparecem no arquivo, agrupadas em BAL, DRE e FLU. Se duas contas gerarem o mesmo nome de coluna, o notebook para com erro.

In [20]:
ORDEM_DEMONSTRACAO = {"BAL": 0, "DRE": 1, "FLU": 2}


def nome_coluna(conta):
    texto = unicodedata.normalize("NFKD", conta).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-zA-Z0-9]+", "_", texto).strip("_").lower()


def ordem_demonstracao(conta):
    return ORDEM_DEMONSTRACAO[conta[:3]]


contas = df["conta"].drop_duplicates().tolist()
contas.sort(key=ordem_demonstracao)

mapa = {conta: nome_coluna(conta) for conta in contas}

assert len(set(mapa.values())) == len(mapa), "Duas contas geraram o mesmo nome de coluna"

df["coluna"] = df["conta"].map(mapa)
print(len(mapa), "contas")

69 contas


## Etapa 8 - Formatando os dados

A tabela larga tem uma linha por cenário-ano e uma coluna por conta.

- O `pivot` não agrega valores: se houver duplicidade de cenário, ano e conta, ele dá erro.
- O cenário passa de `Total Cen_00001` para `Cen_00001`, o mesmo formato usadona planilha da Entrega 1 de Contabilidade e no ranking.
- O ano calendário é `ANO_BASE + ano_num`.

In [21]:
larga = df.pivot(index=["cenario_num", "ano_num"], columns="coluna", values="valor")
larga = larga[list(mapa.values())]
larga = larga.reset_index()
larga.columns.name = None

larga.insert(0, "cenario", "Cen_" + larga["cenario_num"].astype(str).str.zfill(5))
larga.insert(3, "ano_calendario", ANO_BASE + larga["ano_num"])

print("Formato da tabela larga:", larga.shape)
larga.head()

Formato da tabela larga: (14400, 73)


,cenario,cenario_num,ano_num,ano_calendario,bal_total_do_ativo,bal_ativo_circulante,bal_disponivel,bal_contas_a_receber_swap,bal_contas_a_receber_partes_relacionadas,bal_contas_a_receber_clientes,...,flu_tributos,flu_custos,flu_investimentos,flu_entradas,flu_despesas_financeiras,flu_resultado_financeiro,flu_imposto_de_renda_e_contribuicao_social,flu_distribuicao_para_acionista,flu_geracao_de_caixa,flu_saldo_final
0,Cen_00001,1,1,2027,1.063113e+10,1.795721e+09,1.205265e+09,9.568042e+07,584661.466883,4.538710e+08,...,-4.732576e+08,-7.616372e+08,-5.340012e+08,9.162700e+07,-8.220314e+08,-7.304044e+08,-9.344728e+08,-1.532408e+09,4.306498e+08,1.205265e+09
1,Cen_00001,1,2,2028,1.035499e+10,1.567979e+09,1.022427e+09,1.625707e+08,584661.466883,4.055679e+08,...,-5.050062e+08,-7.857755e+08,-3.320350e+08,1.515790e+08,-6.972945e+08,-5.457156e+08,-1.333357e+09,-2.553746e+09,-1.828375e+08,1.022427e+09
2,Cen_00001,1,3,2029,1.081273e+10,2.292584e+09,1.748421e+09,2.248569e+08,584661.466883,4.055679e+08,...,-5.339775e+08,-7.826596e+08,-1.481845e+08,1.097470e+08,-5.984802e+08,-4.887332e+08,-1.374441e+09,-2.069738e+09,7.259936e+08,1.748421e+09
3,Cen_00001,1,4,2030,9.646283e+09,1.655697e+09,1.117390e+09,2.826332e+08,584661.466883,4.055679e+08,...,-5.647615e+08,-1.283996e+09,-8.644133e+07,1.868873e+08,-1.767784e+09,-1.580896e+09,-1.337788e+09,-2.220636e+09,-6.310307e+08,1.117390e+09
4,Cen_00001,1,5,2031,8.067280e+09,8.393786e+08,3.073864e+08,3.398336e+08,584661.466883,4.055679e+08,...,-5.939120e+08,-1.662593e+09,-4.467833e+07,1.179543e+08,-1.643826e+09,-1.525871e+09,-1.334741e+09,-2.460296e+09,-8.100039e+08,3.073864e+08


## Etapa 9 - Conferindo e salvando os dados

Conferências:

1. 1.200 cenários x 12 anos = 14.400 linhas.
2. Ativo + Passivo deve ser ~0 (o passivo vem com sinal negativo; sobra só uma diferença de arredondamento).
3. Valores de um cenário conferem com o CSV original (Cen_00001, Ano 1).
4. As contas que não existem no Ano 12 (encerramento da concessão) ficam vazias; o notebook mostra quantas são.

Depois das conferências, a tabela é gravada em Parquet.

In [22]:
assert larga.shape[0] == 1200 * 12, larga.shape[0]

diferenca = (larga["bal_total_do_ativo"] + larga["bal_total_do_passivo"]).abs()
print("Maior diferença Ativo + Passivo:", diferenca.max())
assert diferenca.max() < 1

ref = larga[(larga["cenario_num"] == 1) & (larga["ano_num"] == 1)].iloc[0]
assert abs(ref["dre_receita"] - 6722729306.39378) < 1
assert abs(ref["dre_ebitda"] - 5188380666.7843) < 1

ano12 = larga[larga["ano_num"] == 12]
print("Contas sem nenhum valor no Ano 12:", int(ano12.isna().all().sum()), "de", len(mapa))
print("Conferências ok")

Maior diferença Ativo + Passivo: 0.025201797485351562
Contas sem nenhum valor no Ano 12: 27 de 69
Conferências ok


In [24]:
destino = os.path.join(PASTA_SAIDA, "base_larga.parquet")
larga.to_parquet(destino, index=False)

print("Salvo em:", os.path.abspath(destino))
print("Tamanho: %.1f MB" % (os.path.getsize(destino) / 1e6))

Salvo em: /Users/harryzhu/Projects/Projeto7/src/Staging/base_larga.parquet
Tamanho: 5.0 MB
